# 03 — Hyperparameter Tuning & Overfitting Check
GridSearchCV with StratifiedKFold(5) to find best params, then train/test F1 comparison to detect overfitting.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data import load_data, split_data
from src.tuning import tune_decision_tree, tune_knn, tune_naive_bayes
from src.models import train_decision_tree, train_knn, train_naive_bayes
from src.evaluate import evaluate

In [2]:
X, y = load_data('../data/heart_failure_clinical_records_dataset.csv')
X_train, X_test, y_train, y_test = split_data(X, y)

## Grid Search Results

In [3]:
dt_params, dt_cv_score = tune_decision_tree(X_train, y_train)
knn_params, knn_cv_score = tune_knn(X_train, y_train)
nb_params, nb_cv_score = tune_naive_bayes(X_train, y_train)

tuning_summary = pd.DataFrame([
    {'Model': 'Decision Tree', 'Best Params': str(dt_params), 'CV F1 (weighted)': round(dt_cv_score, 4)},
    {'Model': 'KNN',           'Best Params': str(knn_params), 'CV F1 (weighted)': round(knn_cv_score, 4)},
    {'Model': 'Naive Bayes',   'Best Params': str(nb_params), 'CV F1 (weighted)': round(nb_cv_score, 4)},
]).set_index('Model')

tuning_summary

,Best Params,CV F1 (weighted)
Model,,
Decision Tree,"{'max_depth': 3, 'min_samples_leaf': 10}",0.6927
KNN,{'n_neighbors': 5},0.6829
Naive Bayes,{'var_smoothing': 1e-11},0.6725


## Overfitting Check — Train F1 vs Test F1
A large gap between train and test F1 indicates overfitting.

In [4]:
rows = []

for train_fn, name in [
    (train_decision_tree, 'Decision Tree'),
    (train_knn,           'KNN'),
    (train_naive_bayes,   'Naive Bayes'),
]:
    pipe = train_fn(X_train, y_train)
    train_results = evaluate(pipe, X_train, y_train, model_name=f'{name} [train]')
    test_results  = evaluate(pipe, X_test,  y_test,  model_name=f'{name} [test]')
    gap = round(train_results['f1'] - test_results['f1'], 4)
    rows.append({
        'Model':      name,
        'Train F1':   round(train_results['f1'], 4),
        'Test F1':    round(test_results['f1'],  4),
        'Gap':        gap,
        'Overfit?':   'yes' if gap > 0.10 else 'no',
    })
    print()

[Decision Tree [train]] Accuracy: 0.8117 | F1: 0.7989
              precision    recall  f1-score   support

           0       0.81      0.94      0.87       162
           1       0.82      0.53      0.65        77

    accuracy                           0.81       239
   macro avg       0.81      0.74      0.76       239
weighted avg       0.81      0.81      0.80       239

[Decision Tree [test]] Accuracy: 0.7833 | F1: 0.7731
              precision    recall  f1-score   support

           0       0.80      0.90      0.85        41
           1       0.71      0.53      0.61        19

    accuracy                           0.78        60
   macro avg       0.76      0.71      0.73        60
weighted avg       0.78      0.78      0.77        60


[KNN [train]] Accuracy: 0.7699 | F1: 0.7453
              precision    recall  f1-score   support

           0       0.77      0.94      0.85       162
           1       0.78      0.40      0.53        77

    accuracy                  

In [5]:
overfit_df = pd.DataFrame(rows).set_index('Model')
overfit_df

,Train F1,Test F1,Gap,Overfit?
Model,,,,
Decision Tree,0.7989,0.7731,0.0258,no
KNN,0.7453,0.6867,0.0586,no
Naive Bayes,0.6693,0.6627,0.0066,no
